# Real Belgian PV: what the forecasts actually do

**Diagnostic analysis of the already-known March–October 2025 development population.**
This notebook uses actual Elia PV and authenticated, existing weather-experiment predictions.
It never fits a model, regenerates forecasts, clips predictions, downloads weather,
or overwrites a frozen experiment. Patterns found here are **hypotheses for a fresh
confirmatory period**, not permission to tune and claim independent gains on this period.

Read in order: understand PV → compare approaches → inspect wins and failures →
examine forecast weather → investigate residuals and complementarity.
Interactive plots support zoom, hover and legend toggles. The date control needs a
live kernel; exported HTML plots need no server or internet. Plot dates remain UTC
instants; day-plot ticks and hover labels explicitly show Brussels local time.

## Local prerequisites
Select **SolarCast — analysis (.venv/weather-smoke)**. From the repository root:
```powershell
.\.venv\weather-smoke\Scripts\python.exe -m pip install -r notebooks/requirements.txt
.\.venv\weather-smoke\Scripts\python.exe -m jupyterlab notebooks/01_real_belgian_forecast_analysis.ipynb
```
Requires the local LightGBM, TSFM and weather-experiment artifact bundles and original Elia snapshot;
these are deliberately not fetched or replaced. Jupyter is optional, not a runtime dependency.

In [ ]:
import hashlib
import json
import os
import sys
import tomllib
from pathlib import Path

import ipywidgets as widgets
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import Markdown, display
from plotly.subplots import make_subplots

ROOT = next(
    (
        p
        for p in (Path.cwd(), *Path.cwd().parents)
        if (p / "pyproject.toml").is_file() and (p / "src/solarcast").is_dir()
    ),
    None,
)
if ROOT is None:
    raise FileNotFoundError(
        "Run this notebook from the SolarCast repository or notebooks directory."
    )
EXP = ROOT / "artifacts/experiments/weather_lightgbm_v1"
PRIOR = ROOT / "artifacts/experiments/tsfm_zero_shot_v1"
PARENT = ROOT / "artifacts/experiments/lightgbm_baseline_v1"
OUT = ROOT / "artifacts/analysis/01_real_belgian_forecast_analysis"
OUT.mkdir(parents=True, exist_ok=True)
TZ = "Europe/Brussels"


def sha(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


def read_json(path):
    if not Path(path).is_file():
        raise FileNotFoundError(
            f"Required local artifact missing: {path}. No network fallback is allowed."
        )
    return json.loads(Path(path).read_text(encoding="utf8"))


def canonical(value):
    return hashlib.sha256(
        json.dumps(value, sort_keys=True, default=str, allow_nan=False).encode()
    ).hexdigest()


PROTECTED = [ROOT / "artifacts/experiments", ROOT / "data", ROOT / "configs/experiments"]


def protected(path):
    if not isinstance(path, (str, bytes, os.PathLike)):
        return False
    p = Path(os.fsdecode(path)).resolve()
    return any(p.is_relative_to(base.resolve()) for base in PROTECTED)


def read_only_offline_audit(event, args):
    if event == "socket.connect":
        address = args[1]
        if isinstance(address, tuple) and address[0] not in ("127.0.0.1", "::1", "localhost"):
            raise RuntimeError(f"External network blocked during diagnostic analysis: {address[0]}")
    if event == "open" and protected(args[0]):
        mode, flags = args[1], args[2]
        writing = isinstance(mode, str) and any(c in mode for c in "wax+")
        writing |= bool(
            isinstance(flags, int)
            and flags & (os.O_WRONLY | os.O_RDWR | os.O_CREAT | os.O_TRUNC | os.O_APPEND)
        )
        if writing:
            raise PermissionError("Frozen experiment/data/configuration writes are blocked.")
    if event in ("os.remove", "os.rmdir", "os.rename"):
        paths = args[:2] if event == "os.rename" else args[:1]
        if any(protected(p) for p in paths):
            raise PermissionError("Frozen artifact mutation is blocked.")


sys.addaudithook(read_only_offline_audit)


def frozen_signatures():
    return {
        str(p.relative_to(ROOT)): sha(p)
        for base in PROTECTED
        if base.exists()
        for p in sorted(base.rglob("*"))
        if p.is_file()
    }


BEFORE = frozen_signatures()
manifest = read_json(EXP / "results_manifest.json")
for name, expected in manifest["files"].items():
    if sha(EXP / name) != expected:
        raise ValueError(f"Frozen weather-experiment artifact checksum mismatch: {name}")
freeze = read_json(EXP / "freeze.json")
assert freeze["identity"] == canonical(freeze["payload"]) == manifest["identity"]
prefit = read_json(EXP / "pre_fit_freeze.json")
assert prefit["identity"] == canonical(prefit["payload"]) == freeze["payload"]["prefit_identity"]
cfg = tomllib.loads((EXP / "config.toml").read_text(encoding="utf8"))
assert sha(PRIOR / "freeze.json") == cfg["tsfm_freeze_sha256"]
assert sha(PRIOR / "results_manifest.json") == cfg["tsfm_results_sha256"]
for name, expected in read_json(PRIOR / "results_manifest.json")["files"].items():
    assert sha(PRIOR / name) == expected, name
for name, expected in read_json(PARENT / "metadata.json")["artifacts_sha256"].items():
    assert sha(PARENT / name) == expected, name
amendment = read_json(EXP / "analysis_amendment.json")
assert amendment["inference_identity"] == freeze["identity"]
assert amendment["original_sources"] == freeze["payload"]["sources"]
assert not amendment["refit_or_reinference"]
assert sha(EXP / "analysis_source_snapshot.zip") == amendment["analysis_archive_sha256"]
for name, expected in amendment["prediction_files"].items():
    assert sha(EXP / name) == expected, name
assert sha(PARENT / "snapshot.json") == prefit["payload"]["snapshot_sha256"]
display(
    Markdown(
        f"**Authenticated:** {len(manifest['files'])} weather-experiment files, original LightGBM/TSFM experiment bundles, "
        f"and {len(BEFORE):,} protected local files. No current-source equality is falsely claimed; "
        "this notebook reads recorded artifacts, including the reporting-only amendment."
    )
)

## 1. What are we forecasting?

National Belgian Elia `ods032.measured`: estimated/upscaled interval-average PV
power in **MW**, at 15-minute interval starts. It is not a single plant, irradiance,
or direct measurement of every Belgian panel. The 11:00 Brussels issue targets the
next local calendar day, including 23/25-hour DST days. PV availability uses a
30-minute safety lag; GFS availability is **assumed initialization + 6 hours**.
Archive publication/as-of fidelity and provider-pretraining overlap remain unverified.

In [ ]:
from solarcast.features import solar_features

selection = read_json(EXP / "selection.json")
WEATHER = selection["model_id"]
BASE = cfg["reference_model"]
PRIMARY = [BASE, "lightgbm", "t0", WEATHER]
ALL = PRIMARY + ["chronos"]
LABEL = {
    BASE: "48h persistence",
    "lightgbm": "History-LightGBM",
    "t0": "t0-gamma",
    WEATHER: "Weather-LightGBM",
    "chronos": "Chronos-2",
}
COLORS = {
    BASE: "#d08720",
    "lightgbm": "#377eb8",
    "t0": "#6a51a3",
    WEATHER: "#218739",
    "chronos": "#bc5090",
}
KEY = ["issue_time", "target_time"]


def csv(name, base=EXP):
    frame = pd.read_csv(base / name)
    for col in ("issue_time", "target_time"):
        if col in frame:
            frame[col] = pd.to_datetime(frame[col], utc=True).astype("datetime64[ns, UTC]")
    return frame


snapshot = read_json(PARENT / "snapshot.json")
parts = []
for shard in snapshot:
    path = (ROOT / shard["file"]).resolve()
    assert path.is_relative_to(ROOT) and sha(path) == shard["sha256"]
    sidecar = read_json(path.with_suffix(".json"))
    assert all(sidecar[k] == shard[k] for k in ("sha256", "start", "end"))
    part = pd.read_csv(path, usecols=["target_time", "pv_generation_mw"])
    part["target_time"] = pd.to_datetime(part.target_time, utc=True).astype("datetime64[ns, UTC]")
    parts.append(part)
history = pd.concat(parts, ignore_index=True).sort_values("target_time").reset_index(drop=True)
assert not history.target_time.duplicated().any()
aligned = csv("aligned_predictions.csv")
assert set(aligned.model_id) == set(PRIMARY)
assert len(aligned) == 4 * 23520 and not aligned.duplicated(KEY + ["model_id"]).any()
truth = aligned.drop_duplicates(KEY)[KEY + ["actual_mw"]].sort_values(KEY).reset_index(drop=True)
assert len(truth) == 23520 and truth.issue_time.nunique() == 245
observed = history.set_index("target_time").pv_generation_mw.reindex(truth.target_time)
assert np.array_equal(observed.to_numpy(float), truth.actual_mw.to_numpy(float))
for model in PRIMARY:
    frame = aligned.loc[aligned.model_id.eq(model)].sort_values(KEY)
    assert pd.MultiIndex.from_frame(frame[KEY]).equals(pd.MultiIndex.from_frame(truth[KEY]))
    assert np.array_equal(frame.actual_mw.to_numpy(float), truth.actual_mw.to_numpy(float))
    assert frame.status.eq("ok").all()
chronos = csv("predictions_chronos.csv")
assert pd.MultiIndex.from_frame(chronos[KEY]).equals(pd.MultiIndex.from_frame(truth[KEY]))
assert np.array_equal(chronos.actual_mw.to_numpy(float), truth.actual_mw.to_numpy(float))
assert chronos.prediction_mw.notna().all() and chronos.status.eq("ok").all()
predictions = pd.concat([aligned, chronos], ignore_index=True)
predictions["local_date"] = predictions.target_time.dt.tz_convert(TZ).dt.strftime("%Y-%m-%d")
predictions["local_hour"] = predictions.target_time.dt.tz_convert(TZ).dt.hour
predictions["error_mw"] = predictions.prediction_mw.astype(float) - predictions.actual_mw.astype(
    float
)
predictions["absolute_error_mw"] = predictions.error_mw.abs()
predictions["squared_error_mw2"] = predictions.error_mw**2
truth["local_date"] = truth.target_time.dt.tz_convert(TZ).dt.strftime("%Y-%m-%d")
truth["local_hour"] = truth.target_time.dt.tz_convert(TZ).dt.hour
truth["month"] = truth.target_time.dt.tz_convert(TZ).dt.strftime("%Y-%m")
weather = csv("test_weather_features.csv")
assert len(weather) == 23520 and not weather.duplicated(KEY).any()
assert pd.MultiIndex.from_frame(weather[KEY]).equals(pd.MultiIndex.from_frame(truth[KEY]))
availability = csv("weather_availability.csv")
test_vintages = availability.loc[availability.split.eq("test")].copy()
assert len(test_vintages) == 245 and test_vintages.status.eq("ok").all()
assert test_vintages.availability_basis.eq("assumed").all()
assert test_vintages.policy_name.eq("gfs_init_plus_6h_v1").all()
assert (pd.to_datetime(test_vintages.available_at, utc=True) <= test_vintages.issue_time).all()
assert (
    pd.to_datetime(test_vintages.reference_time, utc=True) + pd.Timedelta(hours=6)
    == pd.to_datetime(test_vintages.available_at, utc=True)
).all()

solar = solar_features(pd.DatetimeIndex(truth.target_time))
truth["daylight"] = solar.is_daylight.to_numpy(bool)
assert truth.daylight.sum() == 13580
predictions = predictions.merge(truth[KEY + ["daylight"]], on=KEY, validate="many_to_one")
joined = predictions.merge(weather, on=KEY, validate="many_to_one")
daily = csv("daily_metrics.csv")
wins = csv("daily_win_summary.csv")
winners = csv("daily_winners.csv")
hours = csv("by_local_hour.csv")
horizons = csv("horizon_metrics.csv")
months = csv("by_month.csv")
regimes = csv("weather_regimes.csv")
ablation = csv("validation_ablation.csv")
leaderboard = csv("extended_leaderboard.csv")
primary_leaderboard = csv("leaderboard.csv")
daylight_leaderboard = csv("daylight_leaderboard.csv")
assert leaderboard.n_scored.eq(23520).all() and set(leaderboard.model_id) == set(ALL)
baseline = predictions.loc[predictions.model_id.eq(BASE)].sort_values(KEY)
lagged = history.set_index("target_time").pv_generation_mw.reindex(
    pd.DatetimeIndex(baseline.target_time) - pd.Timedelta(hours=48)
)
assert np.array_equal(lagged.to_numpy(float), baseline.prediction_mw.to_numpy(float))
display(
    pd.DataFrame(
        {
            "Source": [
                "Elia snapshot",
                "Primary forecast rows",
                "Chronos forecast",
                "Prepared GFS",
                "Leaderboard/daily diagnostics",
            ],
            "Path": [
                str(PARENT / "snapshot.json"),
                str(EXP / "aligned_predictions.csv"),
                str(EXP / "predictions_chronos.csv"),
                str(EXP / "test_weather_features.csv"),
                str(EXP),
            ],
        }
    )
)

In [ ]:
pio.templates.default = "plotly_white"
pio.renderers.default = "plotly_mimetype+notebook"
FIGURES = []


def named(frame):
    return frame.assign(model=frame.model_id.map(LABEL))


def emit(fig, slug):
    fig.update_layout(
        font=dict(family="Arial", size=13),
        margin=dict(l=65, r=35, t=75, b=65),
        legend_title_text="",
        hovermode="closest",
    )
    fig.write_html(
        OUT / f"{slug}.html", include_plotlyjs="directory", full_html=True, auto_open=False
    )
    FIGURES.append(slug)
    fig.show()
    return None


def local_ticks(fig, times):
    ticks = pd.DatetimeIndex(times).sort_values()[::12]
    fig.update_xaxes(
        tickmode="array",
        tickvals=ticks,
        ticktext=ticks.tz_convert(TZ).strftime("%H:%M %z"),
        title="Belgian local interval start (UTC offsets distinguish DST folds)",
    )


def plot_day(day, include_chronos=False, slug=None):
    selected = predictions.loc[predictions.local_date.eq(str(day))]
    if selected.empty:
        raise ValueError(f"No authenticated forecasts for {day}")
    models = ALL if include_chronos else PRIMARY
    fig = go.Figure()
    actual = selected.drop_duplicates("target_time").sort_values("target_time")
    fig.add_trace(
        go.Scatter(
            x=actual.target_time,
            y=actual.actual_mw,
            mode="lines",
            name="Actual Elia PV",
            line=dict(color="#151515", width=3),
            customdata=actual.target_time.dt.tz_convert(TZ).astype(str),
            hovertemplate="%{customdata}<br>Actual %{y:.1f} MW<extra></extra>",
        )
    )
    for model in models:
        frame = selected.loc[selected.model_id.eq(model)].sort_values("target_time")
        fig.add_trace(
            go.Scatter(
                x=frame.target_time,
                y=frame.prediction_mw,
                mode="lines",
                name=LABEL[model],
                line=dict(color=COLORS[model], width=1.8),
                customdata=frame.target_time.dt.tz_convert(TZ).astype(str),
                hovertemplate="%{customdata}<br>%{y:.1f} MW<extra>%{fullData.name}</extra>",
            )
        )
    local_ticks(fig, actual.target_time)
    fig.update_layout(
        title=f"{day}: actual national PV and raw day-ahead forecasts",
        yaxis_title="PV power (MW)",
        height=480,
    )
    emit(fig, slug or f"day_{day}")
    primary_day = selected.loc[selected.model_id.isin(PRIMARY)]
    day_scores = primary_day.groupby("model_id").squared_error_mw2.mean().pow(0.5)
    winner = day_scores.idxmin()
    weather_peak = float(primary_day.loc[primary_day.model_id.eq(WEATHER), "prediction_mw"].max())
    t0_peak = float(primary_day.loc[primary_day.model_id.eq("t0"), "prediction_mw"].max())
    actual_peak = float(actual.actual_mw.max())
    display(
        Markdown(
            f"**Observed for {day}:** {LABEL[winner]} has the lowest RMSE among the four primary models "
            f"({day_scores[winner]:.2f} MW). Weather RMSE is {day_scores[WEATHER]:.2f} MW versus "
            f"t0's {day_scores['t0']:.2f} MW. Their daily peak-amplitude errors are "
            f"{weather_peak - actual_peak:+.1f} MW and {t0_peak - actual_peak:+.1f} MW, respectively. "
            "These describe errors; they do not establish their weather or fleet-related cause."
        )
    )


def score_plot(table, group, title, slug, metric="rmse_mw"):
    frame = named(table.loc[table.model_id.isin(PRIMARY)])
    fig = px.line(
        frame,
        x=group,
        y=metric,
        color="model",
        markers=True,
        title=title,
        color_discrete_map={LABEL[k]: COLORS[k] for k in PRIMARY},
        labels={metric: "RMSE (MW)", group: group.replace("_", " ")},
    )
    return emit(fig, slug)

## 2. Start with actual Belgian PV

All 23,520 authentic test quarters are shown first, before any model discussion.
The second chart summarizes daily peaks and estimated daily energy; peak changes
can reflect sunlight, clouds, fleet changes, estimation and other factors. They
must not be attributed to a specific weather event without evidence.

In [ ]:
fig = go.Figure(
    go.Scattergl(
        x=truth.target_time,
        y=truth.actual_mw,
        mode="lines",
        name="Actual Elia PV",
        line=dict(color="#151515", width=1),
        customdata=truth.target_time.dt.tz_convert(TZ).astype(str),
        hovertemplate="%{customdata}<br>%{y:.1f} MW<extra></extra>",
    )
)
fig.update_layout(
    title="Real Belgian PV, March–October 2025: every 15-minute interval",
    xaxis_title="UTC time; hover gives Brussels local time",
    yaxis_title="National PV power (MW)",
    xaxis=dict(rangeslider=dict(visible=True)),
    height=450,
)
emit(fig, "01_full_actual_pv")
pv_daily = truth.groupby("local_date").agg(
    peak_mw=("actual_mw", "max"),
    energy_mwh=("actual_mw", lambda s: float(s.sum() * 0.25)),
    zero_quarters=("actual_mw", lambda s: int((s == 0).sum())),
    n_quarters=("actual_mw", "size"),
)
emit(
    px.line(
        pv_daily.reset_index(),
        x="local_date",
        y="peak_mw",
        title="Daily peak actual PV: descriptive seasonal and fleet-related structure",
        labels={"peak_mw": "Daily peak PV (MW)", "local_date": "Belgian target date"},
    ),
    "02_daily_peaks",
)
display(
    truth.groupby("month").agg(
        peak_mw=("actual_mw", "max"),
        mean_power_mw=("actual_mw", "mean"),
        zero_quarters=("actual_mw", lambda s: int(s.eq(0).sum())),
        quarters=("actual_mw", "size"),
    )
)
display(pv_daily.loc[pv_daily.n_quarters.ne(96)])
unusual_profile_days = pv_daily.nsmallest(3, "energy_mwh").index.tolist()
display(
    Markdown(
        f"The daily solar cycle and zero-production stretches are visible directly. "
        f"DST days retain their actual interval counts. The three lowest-energy days in this observed population are "
        f"**{', '.join(unusual_profile_days)}**; this is an outcome-selected diagnostic, not a weather attribution."
    )
)

### A fixed representative week

Use the first complete Monday–Sunday week wholly inside the target period, selected
by calendar alone. This view retains every raw quarter; zoom reveals ramps, dips
and day-to-day amplitude changes. It does not identify their physical causes.

In [ ]:
first_day = pd.Timestamp(truth.local_date.min())
week_start = first_day + pd.Timedelta(days=(7 - first_day.weekday()) % 7)
week_end = week_start + pd.Timedelta(days=7)
week_mask = (truth.local_date >= week_start.strftime("%Y-%m-%d")) & (
    truth.local_date < week_end.strftime("%Y-%m-%d")
)
week = truth.loc[week_mask]
fig = px.line(
    week,
    x="target_time",
    y="actual_mw",
    title=f"Calendar-selected week: {week_start.date()}–{(week_end - pd.Timedelta(days=1)).date()}",
    labels={"target_time": "UTC interval start", "actual_mw": "Actual Belgian PV (MW)"},
)
emit(fig, "03_fixed_week_actual")

## 3. Which approach wins overall?

Load the authenticated extended leaderboard, including Chronos-2. Do not silently
re-score another population. Primary daily-win analyses below retain the original
four-model definition (without Chronos); its presence in an extended table does
not change those winners. All have the same 23,520 pairs here.

In [ ]:
cols = [
    "model_id",
    "mae_mw",
    "rmse_mw",
    "bias_mw",
    "rmse_skill",
    "n_scored",
    "forecast_availability",
]
display(named(leaderboard)[["model", *cols[1:]]].round(4))
display(named(daylight_leaderboard)[["model", *cols[1:]]].round(4))
emit(
    px.bar(
        named(leaderboard),
        x="model",
        y="rmse_mw",
        color="model",
        title="Frozen weather-experiment all-interval RMSE: lower is better",
        color_discrete_map={LABEL[k]: COLORS[k] for k in ALL},
        labels={"rmse_mw": "RMSE (MW)", "model": ""},
    ),
    "04_leaderboard",
)
display(
    Markdown(
        "**Evidence:** Weather-LightGBM leads aggregate RMSE and MAE, including daylight. "
        "That does not mean it wins each day. History-LightGBM and persistence have nearly equal RMSE, "
        "but persistence has lower MAE: average and squared-error metrics emphasize different failures."
    )
)

### Inspect any day: editable variable and optional live-kernel control

The default is the previously fixed 15 June date, not a weather-model success.
Change `selected_date` and rerun, or use the dropdown. Legend clicks hide lines.
Chronos is optional to keep the default chart readable. No forecasts are regenerated.

In [ ]:
selected_date = "2025-06-15"
plot_day(selected_date)


def inspect_day(date, include_chronos):
    plot_day(date, include_chronos=include_chronos, slug="interactive_selected_day")


day_control = widgets.interactive(
    inspect_day,
    date=widgets.Dropdown(
        options=sorted(truth.local_date.unique()), value=selected_date, description="Target day"
    ),
    include_chronos=widgets.Checkbox(value=False, description="Include Chronos-2"),
)
display(day_control)

## 4. When does each model win—and fail?

Best/worst means smallest/largest **stored daily RMSE**, with chronological
tie-breaking. These are explicitly outcome-selected diagnostics, not representative
accuracy evidence. Plot every major model's best and worst case, including cases
that contradict the aggregate winner.

In [ ]:
cases = []
for model in PRIMARY:
    rows = daily.loc[daily.model_id.eq(model)].sort_values("local_date")
    for kind, ascending in (("best", True), ("worst", False)):
        row = rows.sort_values("rmse_mw", ascending=ascending, kind="stable").iloc[0]
        cases.append(
            {
                "model": LABEL[model],
                "kind": kind,
                "local_date": row.local_date,
                "daily_rmse_mw": row.rmse_mw,
            }
        )
case_table = pd.DataFrame(cases)
display(case_table)
for row in case_table.itertuples(index=False):
    display(
        Markdown(
            f"**{row.model}: {row.kind} day**, {row.local_date}, own daily RMSE {row.daily_rmse_mw:.2f} MW."
        )
    )
    plot_day(row.local_date, slug=f"best_worst_{row.kind}_{row.model.replace(' ', '_')}")
display(
    Markdown(
        "Read amplitude, ramp timing and dips separately. A small RMSE day can also be a low-production day; "
        "a worst day alone does not identify a weather or capacity mechanism."
    )
)

In [ ]:
display(named(wins)[["model", "sole_wins", "fractional_days_won", "percent_days_won"]].round(3))
timeline = winners.copy()
assert timeline.n_tied_winners.eq(1).all(), (
    "This display expects the authenticated no-tie result; do not silently collapse ties."
)
timeline["model"] = timeline.winners.map(LABEL)
emit(
    px.scatter(
        timeline,
        x="local_date",
        y="model",
        color="model",
        title="Daily RMSE winner: original four-model definition",
        color_discrete_map={LABEL[k]: COLORS[k] for k in PRIMARY},
        labels={"local_date": "Belgian target date", "model": "Winner"},
    ),
    "05_daily_winner_timeline",
)
emit(
    px.bar(
        named(wins),
        x="model",
        y="percent_days_won",
        color="model",
        title="Fraction of days won, equal day weight",
        color_discrete_map={LABEL[k]: COLORS[k] for k in PRIMARY},
        labels={"percent_days_won": "Days won (%)", "model": ""},
    ),
    "06_daily_win_shares",
)
score_plot(daily, "local_date", "Daily RMSE across the entire development period", "07_daily_rmse")
display(
    Markdown(
        "**Evidence:** weather wins 94/245 days (38.37%), t0 62, persistence 50 and history-LightGBM 39. "
        "The overall winner does not win a majority. Episodic errors and differently sized failures matter."
    )
)

## 5. Where does weather help?

Use the original time-of-day, horizon and monthly definitions. A fixed 11:00 issue
makes horizon and target local time strongly related; these plots cannot isolate
a causal lead-time effect. Zero-reference-error night bins have undefined skill.

In [ ]:
score_plot(hours, "local_hour", "RMSE by Belgian local target hour", "08_local_hour_rmse")
score_plot(
    horizons,
    "horizon_6h",
    "RMSE by original six-hour forecast-horizon bin (left edge)",
    "09_horizon_rmse",
)
score_plot(months, "month", "Monthly RMSE, March–October 2025", "10_monthly_rmse")
monthly_rank = months.pivot(index="month", columns="model_id", values="rmse_mw")
display(monthly_rank.rename(columns=LABEL).round(2))
display(pd.DataFrame({"monthly_RMSE_winner": monthly_rank.idxmin(axis=1).map(LABEL)}))
display(
    Markdown(
        "The aggregate largest weather gain is in the 24–30h bin, broadly next-day midday. "
        "Persistence and t0 remain strong during low-production hours; weather is worse than history-LightGBM "
        "at 21:00 in the saved local-hour table. Eight months are not evidence of universal seasonal superiority."
    )
)

### Is midday improvement broad, or concentrated in a few days?

**Additional exploratory population:** interval starts in [10:00,15:00) Brussels
(10:00–14:45). This does not replace weather-experiment's primary scores. Positive reduction
below means the weather model has smaller daily midday RMSE. A descriptive top-decile
contribution is calculated from signed SSE reductions, not used to select features.

In [ ]:
mid = predictions.loc[predictions.model_id.isin(PRIMARY) & predictions.local_hour.between(10, 14)]
mid_rmse = mid.groupby(["local_date", "model_id"]).squared_error_mw2.mean().pow(0.5).unstack()
mid_delta = pd.DataFrame(
    {
        "local_date": mid_rmse.index,
        "Weather gain over history (MW)": (mid_rmse.lightgbm - mid_rmse[WEATHER]).to_numpy(),
        "Weather gain over t0 (MW)": (mid_rmse.t0 - mid_rmse[WEATHER]).to_numpy(),
    }
)
emit(
    px.line(
        mid_delta,
        x="local_date",
        y=mid_delta.columns[1:],
        title="Daily midday RMSE reduction from weather: positive is better",
        labels={
            "value": "RMSE reduction (MW)",
            "local_date": "Belgian target date",
            "variable": "Comparison",
        },
    ),
    "11_midday_daily_gain",
)
mid_sse = mid.groupby(["local_date", "model_id"]).squared_error_mw2.sum().unstack()
gain = mid_sse.lightgbm - mid_sse[WEATHER]
top_count = int(np.ceil(0.1 * len(gain)))
mid_summary = {
    "days": len(mid_rmse),
    "weather_better_than_history_days": int((mid_rmse[WEATHER] < mid_rmse.lightgbm).sum()),
    "weather_better_than_t0_days": int((mid_rmse[WEATHER] < mid_rmse.t0).sum()),
    "top_decile_days": top_count,
    "top_decile_share_of_net_SSE_reduction": float(gain.nlargest(top_count).sum() / gain.sum()),
}
display(pd.Series(mid_summary))
display(
    Markdown(
        f"Weather improves midday RMSE on **{mid_summary['weather_better_than_history_days']}/{len(mid_rmse)} days** "
        f"versus history-LightGBM, and **{mid_summary['weather_better_than_t0_days']}/{len(mid_rmse)}** versus t0. "
        f"The top {top_count} gain days contribute **{mid_summary['top_decile_share_of_net_SSE_reduction']:.1%}** "
        "of the *net* midday SSE reduction; negative-gain days offset it. This separates frequency from magnitude, not causality."
    )
)

### Bring in the already selected GFS forecasts

Surface DSWRF is correctly reconstructed into hourly W/m² and repeated into quarters;
cloud is instantaneous entire-atmosphere TCDC interpolated to interval means, in
fraction; temperature is °C. Seven approved features were cached, but validation
selected radiation + cloud (six) for the final 19-feature model. No observations or
reanalysis are introduced. There is no approved single national GHI mean: plots
label the west/central/east bands explicitly instead of inventing capacity weights.

In [ ]:
def radiation_profile(day, slug):
    a = truth.loc[truth.local_date.eq(day)]
    w = weather.merge(a[KEY], on=KEY, validate="one_to_one").sort_values("target_time")
    fig = make_subplots(specs=[[{"secondary_y": True}]])
    fig.add_trace(
        go.Scatter(
            x=a.target_time,
            y=a.actual_mw,
            name="Actual national PV (MW)",
            line=dict(color="black", width=3),
        ),
        secondary_y=False,
    )
    for region, color in (("west", "#d08720"), ("central", "#377eb8"), ("east", "#bc5090")):
        col = f"nwp_ghi_{region}_mean_w_m2"
        fig.add_trace(
            go.Scatter(
                x=w.target_time,
                y=w[col],
                name=f"Forecast {region} GHI (W/m²)",
                line=dict(color=color),
            ),
            secondary_y=True,
        )
    local_ticks(fig, a.target_time)
    fig.update_yaxes(title_text="Actual national PV (MW)", secondary_y=False)
    fig.update_yaxes(title_text="Forecast regional GHI (W/m²)", secondary_y=True)
    fig.update_layout(
        title=f"{day}: different quantities—forecast regional radiation versus actual national power",
        height=470,
    )
    return emit(fig, slug)


for day in ("2025-03-15", "2025-06-15", "2025-09-15"):
    radiation_profile(day, f"radiation_{day}")
rad_scatter = joined.loc[joined.model_id.eq(WEATHER) & joined.daylight].copy()
rad_scatter["month"] = rad_scatter.target_time.dt.tz_convert(TZ).dt.strftime("%Y-%m")
emit(
    px.scatter(
        rad_scatter,
        x="nwp_ghi_central_mean_w_m2",
        y="actual_mw",
        color="month",
        opacity=0.25,
        render_mode="webgl",
        title="Daylight only: forecast central-band GHI and actual national PV",
        labels={
            "nwp_ghi_central_mean_w_m2": "Forecast central-band GHI (W/m²)",
            "actual_mw": "Actual national PV (MW)",
        },
    ),
    "12_radiation_actual_scatter",
)
display(
    Markdown(
        "**Evidence:** radiation supplies a future production-related signal unavailable to history-only models. "
        "Scatter and amplitude mismatches remain: coarse regional irradiance and fleet-level PV are different "
        "quantities, with imperfect forecasts and changing fleet/estimation effects. The plot does not establish a fixed physical conversion."
    )
)

In [ ]:
for regime_variable in ("cloud_group", "spatial_variability"):
    table = regimes.loc[regimes.regime_variable.eq(regime_variable)].copy()
    table["population"] = np.where(table.daylight_only, "Daylight", "All intervals")
    emit(
        px.bar(
            named(table),
            x="regime",
            y="rmse_mw",
            color="model",
            barmode="group",
            facet_col="population",
            title=f"Frozen forecast regimes: {regime_variable}",
            color_discrete_map={LABEL[k]: COLORS[k] for k in PRIMARY},
            labels={"rmse_mw": "RMSE (MW)", "regime": "Forecast-only group"},
        ),
        f"13_{regime_variable}",
    )
display(ablation[["feature_set", "mae_mw", "rmse_mw", "n_scored"]].round(3))
display(
    Markdown(
        "**Why weather appears to help—evidence versus hypotheses:** validation radiation gives most of the gain; "
        "cloud adds a smaller increment, while temperature was not selected. Saved midday scores and forecast-regime tables "
        "show where that fixed model improves. The largest reduction over history-LightGBM is in low forecast cloud; "
        "the largest reduction over t0 is in high forecast cloud. **Hypotheses**, not established causes, include radiation "
        "information resolving an amplitude mismatch or history being less informative about changing future conditions. "
        "Actual weather observations, capacity vintages and controlled fresh-period tests are needed to attribute causes. "
        "Spatial cloud standard deviation is forecast spatial heterogeneity, not observed or temporal variability."
    )
)

## 6. Where does t0 help—and where does persistence still work?

Pairwise t0 wins below are **not** the same as sole four-model daily victories.
Examples are the chronological first, middle and last eligible day—no search for
the biggest winning margin. Inspect actual profiles and the contemporaneously
available forecast radiation/cloud, but state when these inputs cannot explain an error.

In [ ]:
daily_matrix = daily.pivot(index="local_date", columns="model_id", values="rmse_mw").sort_index()
t0_better_days = daily_matrix.index[daily_matrix.t0 < daily_matrix[WEATHER]].tolist()


def calendar_examples(days, count=3):
    if not days:
        return []
    return list(
        dict.fromkeys(
            [
                days[i]
                for i in np.linspace(0, len(days) - 1, min(count, len(days))).round().astype(int)
            ]
        )
    )


t0_examples = calendar_examples(t0_better_days)
weather_daily = weather.assign(
    local_date=weather.target_time.dt.tz_convert(TZ).dt.strftime("%Y-%m-%d")
)
weather_daily = weather_daily.groupby("local_date").agg(
    mean_forecast_cloud=("nwp_cloud_belgium_mean_fraction", "mean"),
    mean_forecast_spatial_cloud_std=("nwp_cloud_belgium_spatial_std_fraction", "mean"),
    central_ghi_peak_w_m2=("nwp_ghi_central_mean_w_m2", "max"),
    within_day_forecast_cloud_std=("nwp_cloud_belgium_mean_fraction", "std"),
)
display(
    Markdown(
        f"t0 has lower daily RMSE than Weather-LightGBM on **{len(t0_better_days)}/245** days. "
        f"Calendar-spaced examples: **{', '.join(t0_examples)}**."
    )
)
display(weather_daily.join(daily_matrix).loc[t0_examples].round(3))
for day in t0_examples:
    plot_day(day, slug=f"t0_pairwise_example_{day}")
    radiation_profile(day, f"t0_radiation_example_{day}")
display(
    Markdown(
        "Compare timing, peak amplitude and dips rather than inventing a cloud event. Forecast GHI can disagree "
        "with PV while both NWP and the PV mapping contribute error. A good t0 day does not reveal what its pretrained "
        "representation learned. These artifacts alone generally cannot separate GFS error, mapping error, fleet effects and chance."
    )
)

In [ ]:
persist_days = winners.loc[winners.winners.eq(BASE), "local_date"].tolist()
persist_examples = calendar_examples(persist_days)
profile_rows = []
for day, rows in predictions.loc[predictions.model_id.eq(BASE)].groupby("local_date", sort=True):
    a, lag = rows.actual_mw.to_numpy(float), rows.prediction_mw.to_numpy(float)
    profile_rows.append(
        {
            "local_date": day,
            "profile_48h_pearson": np.corrcoef(a, lag)[0, 1],
            "actual_minus_lag_peak_mw": float(a.max() - lag.max()),
            "persistence_wins": day in persist_days,
        }
    )
profiles = pd.DataFrame(profile_rows).set_index("local_date").join(weather_daily)
display(
    profiles.groupby("persistence_wins").agg(
        days=("profile_48h_pearson", "size"),
        median_profile_correlation=("profile_48h_pearson", "median"),
        median_peak_difference_mw=("actual_minus_lag_peak_mw", "median"),
        mean_forecast_cloud=("mean_forecast_cloud", "mean"),
        within_day_forecast_cloud_std=("within_day_forecast_cloud_std", "mean"),
    )
)
win_months = pd.crosstab(
    pd.to_datetime(winners.local_date).dt.strftime("%Y-%m"), winners.winners
).rename(columns=LABEL)
display(win_months)
emit(
    px.box(
        profiles.reset_index(),
        x="persistence_wins",
        y="profile_48h_pearson",
        points="all",
        title="Actual versus exact 48h historical profile: outcome-selected baseline wins",
        labels={
            "persistence_wins": "Persistence is four-model daily winner",
            "profile_48h_pearson": "Within-day profile Pearson correlation",
        },
    ),
    "14_persistence_shape_similarity",
)
for day in persist_examples:
    plot_day(day, slug=f"persistence_example_{day}")
display(
    Markdown(
        "The persistence line is verified against actual PV exactly 48 elapsed UTC hours earlier, including DST. "
        "High profile similarity on winning days is partly expected from selecting on baseline error, not independent evidence "
        "of predictable weather stability. Within-day cloud spread above is a newly labelled forecast diagnostic, not the "
        "frozen spatial-variability regime and not an observed-weather measure."
    )
)

## 7. How do models fail? Bias, negatives, shape and residual structure

Keep signed errors `prediction − actual`, both over- and underforecasting. All raw
negative outputs remain. Additional dawn/dusk labels below are astronomical
descriptions, not new scoring masks or adjustments to forecasts.

In [ ]:
primary_rows = predictions.loc[predictions.model_id.isin(PRIMARY)]
bias_table = (
    primary_rows.groupby("model_id")
    .error_mw.agg(
        mean_error_mw="mean",
        median_error_mw="median",
        overforecast_fraction=lambda s: float((s > 0).mean()),
        underforecast_fraction=lambda s: float((s < 0).mean()),
        p05_mw=lambda s: s.quantile(0.05),
        p95_mw=lambda s: s.quantile(0.95),
    )
    .rename(index=LABEL)
)
display(bias_table.round(3))
emit(
    px.histogram(
        named(primary_rows),
        x="error_mw",
        color="model",
        nbins=120,
        barmode="overlay",
        opacity=0.45,
        histnorm="probability density",
        title="Raw signed-error distributions on the primary common population",
        color_discrete_map={LABEL[k]: COLORS[k] for k in PRIMARY},
        labels={"error_mw": "Prediction − actual (MW)"},
    ),
    "15_signed_error_distribution",
)
elevation = solar.solar_apparent_elevation_deg.to_numpy(float)
phase_table = truth[KEY].copy()
phase_table["phase"] = np.select(
    [elevation > 0, (elevation > -6) & (truth.local_hour.to_numpy() < 12), elevation > -6],
    ["Daylight", "Dawn twilight", "Dusk twilight"],
    default="Night (<=−6°)",
)
negatives = predictions.merge(phase_table, on=KEY, validate="many_to_one")
negatives = negatives.loc[negatives.prediction_mw.lt(0)]
negative_table = (
    negatives.groupby("model_id")
    .prediction_mw.agg(count="size", minimum_mw="min", median_negative_mw="median")
    .reindex(ALL)
)
negative_table["count"] = (
    negative_table["count"].fillna(0).astype(int)
)  # Count display only; no prediction is filled.
display(negative_table.rename(index=LABEL))
negative_phases = (
    negatives.groupby(["model_id", "phase"]).size().rename("negative_quarters").reset_index()
)
emit(
    px.bar(
        named(negative_phases),
        x="phase",
        y="negative_quarters",
        color="model",
        barmode="group",
        title="When do negative predictions occur? No clipping",
        color_discrete_map={LABEL[k]: COLORS[k] for k in ALL},
    ),
    "16_negative_prediction_phases",
)
emit(
    px.scatter(
        named(negatives),
        x="local_hour",
        y="prediction_mw",
        color="model",
        opacity=0.3,
        render_mode="webgl",
        title="Negative magnitude versus Belgian local hour",
        labels={"prediction_mw": "Raw negative prediction (MW)"},
        color_discrete_map={LABEL[k]: COLORS[k] for k in ALL},
    ),
    "17_negative_magnitudes",
)
display(
    Markdown(
        "Small aggregate bias can conceal opposite errors across daylight, night, seasons and regimes. "
        "Negative outputs are particularly common for t0 but must be read with their magnitudes and times, not just counts. "
        "These plots neither correct them nor measure the benefit of hypothetical clipping."
    )
)

In [ ]:
# Per-day shape summaries compare timing and amplitude without fitting any correction.
shape_rows = []
for (day, model), frame in primary_rows.groupby(["local_date", "model_id"], sort=True):
    frame = frame.sort_values("target_time")
    a, p = frame.actual_mw.to_numpy(float), frame.prediction_mw.to_numpy(float)

    def centroid(values):
        # Raw signed predictions retained; statistic undefined if their sum is nonpositive.
        return (
            np.nan
            if values.sum() <= 0
            else float(np.dot(np.arange(len(values)) * 0.25, values) / values.sum())
        )

    shape_rows.append(
        {
            "local_date": day,
            "model_id": model,
            "peak_error_mw": float(p.max() - a.max()),
            "peak_timing_error_hours": float((np.argmax(p) - np.argmax(a)) * 0.25),
            "energy_centroid_error_hours": centroid(p) - centroid(a),
            "profile_correlation": float(np.corrcoef(a, p)[0, 1]),
        }
    )
shapes = pd.DataFrame(shape_rows)
display(named(shapes.loc[shapes.local_date.isin(["2025-03-15", "2025-06-15", "2025-09-15"])]))
emit(
    px.box(
        named(shapes),
        x="model",
        y="peak_error_mw",
        title="Daily peak-amplitude errors: raw forecasts",
        color="model",
        color_discrete_map={LABEL[k]: COLORS[k] for k in PRIMARY},
        labels={"peak_error_mw": "Predicted − actual daily peak (MW)"},
    ),
    "18_peak_shape_errors",
)
for day in ("2025-03-15", "2025-09-15"):
    plot_day(day, slug=f"shape_fixed_{day}")
display(
    Markdown(
        "Peak-time ties use the earliest maximum; energy-centroid errors summarize asymmetry but retain signed "
        "predictions and are not a physical validation of negative power. The fixed day plots expose ramps, step-like model "
        "responses and missed dips that RMSE alone cannot describe. Shape summaries are new diagnostics, not training targets."
    )
)

In [ ]:
for x, title in (
    ("actual_mw", "Actual PV (MW)"),
    ("local_hour", "Belgian local hour"),
    ("nwp_ghi_central_mean_w_m2", "Forecast central-band GHI (W/m²)"),
    ("nwp_cloud_belgium_mean_fraction", "Forecast national cloud fraction"),
):
    frame = joined.loc[joined.model_id.isin(PRIMARY)]
    fig = px.scatter(
        named(frame),
        x=x,
        y="error_mw",
        color="model",
        facet_col="model",
        facet_col_wrap=2,
        opacity=0.12,
        render_mode="webgl",
        height=650,
        color_discrete_map={LABEL[k]: COLORS[k] for k in PRIMARY},
        title=f"Residual structure against {title}: all primary common quarters",
        labels={x: title, "error_mw": "Prediction − actual (MW)"},
    )
    fig.add_hline(y=0, line_width=1, line_color="black")
    emit(fig, f"residual_{x}")
display(
    Markdown(
        "Residual structure may motivate hypotheses about amplitude extrapolation, time-dependent mapping or "
        "forecast covariates, but the plots cannot identify the cause. No discovered pattern is turned into a feature, "
        "correction or tuned parameter on these already-known targets."
    )
)

## 8. Are errors complementary? Diagnostic ensemble opportunity only

Use the same 23,520 common issue-target pairs as weather-experiment, and verify Pearson
values against its stored error-correlation table. No averages of model forecasts,
ensemble weights, oracle forecasts or ensemble performance claims are produced.

In [ ]:
error_wide = primary_rows.pivot(index=KEY, columns="model_id", values="error_mw").sort_index()
stored_corr = pd.read_csv(EXP / "error_correlations.csv", index_col=0)
corr = error_wide.corr()
assert np.allclose(corr.loc[stored_corr.index, stored_corr.columns], stored_corr, atol=1e-12)
display(corr.rename(index=LABEL, columns=LABEL).round(4))
for x, y in ((WEATHER, "t0"), (WEATHER, "lightgbm"), ("t0", BASE)):
    frame = error_wide[[x, y]].reset_index()
    fig = px.scatter(
        frame,
        x=x,
        y=y,
        opacity=0.18,
        render_mode="webgl",
        title=f"Signed errors: {LABEL[x]} versus {LABEL[y]} (r={corr.loc[x, y]:.3f})",
        labels={x: f"{LABEL[x]} error (MW)", y: f"{LABEL[y]} error (MW)"},
    )
    fig.add_hline(y=0, line_color="black")
    fig.add_vline(x=0, line_color="black")
    emit(fig, f"error_scatter_{PRIMARY.index(x)}_{PRIMARY.index(y)}")
absolute = error_wide[[WEATHER, "t0"]].abs()
fig = px.scatter(
    absolute,
    x=WEATHER,
    y="t0",
    opacity=0.18,
    render_mode="webgl",
    title="Complementarity: below diagonal t0 is closer; above diagonal weather is closer",
    labels={WEATHER: "Weather-LightGBM absolute error (MW)", "t0": "t0 absolute error (MW)"},
)
limit = float(absolute.to_numpy().max())
fig.add_shape(type="line", x0=0, y0=0, x1=limit, y1=limit, line=dict(dash="dash", color="black"))
emit(fig, "19_absolute_error_complementarity")
pairwise_quarters = float(absolute.t0.lt(absolute[WEATHER]).mean())
upper = absolute[WEATHER].ge(absolute[WEATHER].quantile(0.75))
upper_better = float(absolute.loc[upper, "t0"].lt(absolute.loc[upper, WEATHER]).mean())
display(
    Markdown(
        f"t0 is closer on **{pairwise_quarters:.1%}** of all common quarters. In the outcome-selected upper "
        f"quartile of weather absolute errors, t0 is closer on **{upper_better:.1%}**. Signed-error correlation is "
        f"**{corr.loc[WEATHER, 't0']:.4f}**: moderate, not zero. This supports a fresh-period ensemble *experiment*, "
        "not an estimate of attainable blend skill or a rule available in advance about which forecast to trust."
    )
)

day_keys = pd.MultiIndex.from_frame(truth.loc[truth.daylight, KEY])
day_mask = absolute.index.isin(day_keys)
day_pairwise = float(absolute.loc[day_mask, "t0"].lt(absolute.loc[day_mask, WEATHER]).mean())
night_pairwise = float(absolute.loc[~day_mask, "t0"].lt(absolute.loc[~day_mask, WEATHER]).mean())
tail_table = primary_rows.groupby(["model_id", "daylight"]).absolute_error_mw.agg(
    median_absolute_mw="median",
    p95_absolute_mw=lambda s: s.quantile(0.95),
    p99_absolute_mw=lambda s: s.quantile(0.99),
    maximum_absolute_mw="max",
)
display(tail_table.rename(index=LABEL, level=0).round(2))
display(
    Markdown(
        f"**Do not confuse frequency with error magnitude:** t0 is closer on {day_pairwise:.1%} of daylight "
        f"quarters, versus {night_pairwise:.1%} of non-daylight quarters. Its all-quarter count is strongly influenced "
        "by night. RMSE and MAE weight error sizes, so winning more individual quarters does not guarantee a lower "
        "aggregate score. The displayed tails characterize this distinction without fitting a correction."
    )
)

# What We Learned

**This is an already-known development/test population. Do not modify models using
these discoveries and claim independent improvement on the same period.**

- **Overall:** Weather-LightGBM leads aggregate RMSE/MAE, but wins only 94/245 days.
- **Weather:** the frozen radiation/cloud model gains most around midday. The
  comparator matters: gains versus history-LightGBM are largest in low forecast
  cloud, whereas gains versus t0 are largest in high forecast cloud. Forecast
  variables are useful information, not verified future weather observations.
- **TSFM:** t0 still wins many pairwise and sole-win days; inspect the calendar-spaced
  examples above. Available artifacts do not reliably distinguish NWP error,
  learned PV-mapping error, fleet effects or chance as their cause.
- **Persistence:** exact 48h historical profiles can remain very competitive,
  including the fixed June example. Similarity on winning days is partly a
  consequence of selecting on forecast error, not proof of predictable clear skies.
- **Errors:** bias, peak-amplitude errors, raw negatives and residual structure
  vary across time and conditions. Aggregate bias alone does not establish calibration.
- **Ensemble:** moderate signed-error correlation and pairwise counterexamples
  justify testing complementarity on a fresh period. No weights or ensemble were fitted.
- **Next experiment—hypotheses only:** freeze a fresh confirmatory protocol before
  testing whether forecast radiation/cloud benefits persist; whether t0 adds value
  in complementary cases; whether amplitude/nonstationarity effects persist after
  obtaining defensible capacity/vintage evidence; and whether uncertainty forecasts
  are calibrated. Archive publication assumptions and unknown pretraining overlap
  remain limitations. No new modeling experiment starts in this notebook.

In [ ]:
display(
    Markdown(
        f"### Quantified observations from this execution\n"
        f"Weather leads monthly aggregate RMSE in **{int(monthly_rank.idxmin(axis=1).eq(WEATHER).sum())}/8 months**, "
        f"but t0 beats it in daily RMSE on **{len(t0_better_days)}/245 days** (versus 62 sole four-model wins). "
        f"Weather improves midday RMSE over history on **{mid_summary['weather_better_than_history_days']}/245 days**; "
        f"the top 25 improvement days supply **{mid_summary['top_decile_share_of_net_SSE_reduction']:.1%}** of net midday SSE reduction. "
        "Thus the gain is broad in frequency but uneven in magnitude. "
        "Persistence-winning days have higher observed 48h-profile similarity; that is partly outcome selection. "
        "Their within-day forecast cloud spread is not appreciably lower in this descriptive comparison, "
        "so this notebook does **not** establish a stable-weather explanation. "
        "These are observations for a future frozen protocol, not modifications to today's models."
    )
)

In [ ]:
assert frozen_signatures() == BEFORE, "A protected artifact changed during notebook execution."
assert not {"lightgbm", "torch", "t0", "chronos"}.intersection(sys.modules), (
    "Unexpected model runtime import."
)
summary = {
    "primary_pairs": len(truth),
    "origins": int(truth.issue_time.nunique()),
    "daylight_pairs": int(truth.daylight.sum()),
    "protected_files_unchanged": len(BEFORE),
    "external_network_blocked": True,
    "model_runtimes_imported": False,
    "t0_pairwise_better_days": len(t0_better_days),
    "midday": mid_summary,
    "quarter_fraction_t0_closer_than_weather": pairwise_quarters,
    "weather_upper_error_quartile_fraction_t0_closer": upper_better,
    "daylight_fraction_t0_closer": day_pairwise,
    "nondaylight_fraction_t0_closer": night_pairwise,
    "interactive_figures": len(FIGURES),
    "figure_files": FIGURES,
}
(OUT / "notebook_summary.json").write_text(json.dumps(summary, indent=2), encoding="utf8")
bias_table.to_csv(OUT / "bias_diagnostics.csv")
negative_table.rename(index=LABEL).to_csv(OUT / "negative_prediction_diagnostics.csv")
case_table.to_csv(OUT / "best_worst_cases.csv", index=False)
shapes.to_csv(OUT / "shape_diagnostics.csv", index=False)
mid_delta.to_csv(OUT / "midday_daily_diagnostics.csv", index=False)
profiles.to_csv(OUT / "persistence_profile_diagnostics.csv")
display(pd.Series(summary).drop("figure_files"))
display(
    Markdown(
        f"**Verified read-only finish:** {len(BEFORE):,} protected files unchanged. "
        f"All derived figures/tables are under `{OUT.relative_to(ROOT)}`. No model runtime was imported; "
        "external connections and frozen-file writes were blocked throughout. Notebook outputs describe real local artifacts only."
    )
)
negative_phases.to_csv(OUT / "negative_phases.csv", index=False)
weather_daily.to_csv(OUT / "day_forecast_weather_diagnostics.csv")
tail_table.to_csv(OUT / "error_tail_diagnostics.csv")
(OUT / "protected_file_hashes.json").write_text(json.dumps(BEFORE, indent=2), encoding="utf8")